<h1 style="color:#1f77b4; font-family:'Times New Roman';">
<b>Transformer Block From Scratch</b>
</h1>
<div style="font-family:'Times New Roman';">
Building one decoder block in torch out of plain functions, multi head attention with the heads done by reshaping, then the feed forward part, residuals and layer norm. At the end i test the causal mask properly instead of assuming it works.
</div>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)

D = 64          # model width
H = 4           # heads
T = 10          # sequence length
d_head = D // H
print("each head gets", d_head, "of the", D, "dimensions")

each head gets 16 of the 64 dimensions


## multi head attention by reshaping

one big matrix produces Q, K and V for all heads together, then the last dimension is split into `(heads, d_head)` and moved so the heads look like a batch dimension. thats the trick, no loop over heads.

In [2]:
W_q = nn.Linear(D, D, bias=False)
W_k = nn.Linear(D, D, bias=False)
W_v = nn.Linear(D, D, bias=False)
W_o = nn.Linear(D, D, bias=False)

def split_heads(x):
    n, T, _ = x.shape
    return x.view(n, T, H, d_head).transpose(1, 2)     # (n, H, T, d_head)

def merge_heads(x):
    n, H_, T, dh = x.shape
    return x.transpose(1, 2).contiguous().view(n, T, H_ * dh)

x = torch.randn(2, T, D)
print("input        :", tuple(x.shape))
print("after split  :", tuple(split_heads(W_q(x)).shape), " (heads act like extra batch)")
print("after merge  :", tuple(merge_heads(split_heads(W_q(x))).shape))

input        : (2, 10, 64)
after split  : (2, 4, 10, 16)  (heads act like extra batch)
after merge  : (2, 10, 64)


In [3]:
def multi_head_attention(x, mask=None):
    q = split_heads(W_q(x))
    k = split_heads(W_k(x))
    v = split_heads(W_v(x))

    scores = q @ k.transpose(-2, -1) / d_head ** 0.5     # (n, H, T, T)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    weights = torch.softmax(scores, dim=-1)
    out = weights @ v
    return W_o(merge_heads(out)), weights

## the causal mask
lower triangle of ones. position t can see everything upto t and nothing after.

In [4]:
mask = torch.tril(torch.ones(T, T)).view(1, 1, T, T)
print(mask[0,0].int().numpy())

[[1 0 0 0 0 0 0 0 0 0]
 [1 1 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 0 0 0 0 0]
 [1 1 1 1 0 0 0 0 0 0]
 [1 1 1 1 1 0 0 0 0 0]
 [1 1 1 1 1 1 0 0 0 0]
 [1 1 1 1 1 1 1 0 0 0]
 [1 1 1 1 1 1 1 1 0 0]
 [1 1 1 1 1 1 1 1 1 0]
 [1 1 1 1 1 1 1 1 1 1]]


In [6]:
with torch.no_grad():
    out, w = multi_head_attention(x, mask)
print("output :", tuple(out.shape))
print("attention weights :", tuple(w.shape), " = (batch, heads, query, key)")
print()
print("row 3 of head 0, everything after position 3 should be zero :")
print(torch.round(w[0,0,3], decimals=3).numpy())

output : (2, 10, 64)
attention weights : (2, 4, 10, 10)  = (batch, heads, query, key)

row 3 of head 0, everything after position 3 should be zero :
[0.203 0.227 0.344 0.225 0.    0.    0.    0.    0.    0.   ]


## the rest of the block

feed forward is `D -> 4D -> D` with a GELU in between, applied to each position separately. then the two sub layers get wrapped in residual + layer norm.

using the pre norm version, `x + sublayer(norm(x))`, because it trains more easily than the post norm in the original paper.

In [14]:
ff = nn.Sequential(
    nn.Linear(D, 4 * D),
    nn.GELU(),
    nn.Linear(4 * D, D),
)

norm1 = nn.LayerNorm(D)
norm2 = nn.LayerNorm(D)

def block(x, mask):
    a, weights = multi_head_attention(norm1(x), mask)
    x = x + a                        # residual 1
    x = x + ff(norm2(x))             # residual 2
    return x, weights

with torch.no_grad():
    y, w = block(x, mask)
print("in :", tuple(x.shape), " out :", tuple(y.shape), " same shape, so blocks can be stacked")

in : (2, 10, 64)  out : (2, 10, 64)  same shape, so blocks can be stacked


## test 1, is the mask really working

instead of trusting the picture, i change a token at the **end** of the sequence and check that the outputs at earlier positions do not move. if they do, information is leaking backwards from the future.

In [8]:
x1 = torch.randn(1, T, D)
x2 = x1.clone()
x2[0, -1] = torch.randn(D)            # change only the last token

with torch.no_grad():
    y1, _ = block(x1, mask)
    y2, _ = block(x2, mask)

diff = (y1 - y2).abs().max(dim=-1).values[0]
for t in range(T):
    print("position", t, " max change :", format(diff[t].item(), '.6f'))

position 0  max change : 0.000000
position 1  max change : 0.000000
position 2  max change : 0.000000
position 3  max change : 0.000000
position 4  max change : 0.000000
position 5  max change : 0.000000
position 6  max change : 0.000000
position 7  max change : 0.000000
position 8  max change : 0.000000
position 9  max change : 4.017024


every position except the last one is unchanged to the last decimal. the mask holds.

running the same test without a mask, where it should leak:

In [10]:
with torch.no_grad():
    y1n, _ = block(x1, None)
    y2n, _ = block(x2, None)

diff_n = (y1n - y2n).abs().max(dim=-1).values[0]
print("without mask, change at position 0 :", format(diff_n[0].item(), '.6f'))
print("with    mask, change at position 0 :", format(diff[0].item(), '.6f'))

without mask, change at position 0 : 0.151433
with    mask, change at position 0 : 0.000000


without the mask a change in the last token moves position 0 as well, wich would be cheating in a language model, it would be looking at the answer.

## test 2, do the residuals help the gradient

stacking 8 blocks and measuring the gradient that reaches the first one, with and without the residual connections.

In [11]:
def deep_stack(x, mask, use_residual, n_layers=8):
    for i in range(n_layers):
        a, _ = multi_head_attention(norm1(x), mask)
        x = x + a if use_residual else a
        f = ff(norm2(x))
        x = x + f if use_residual else f
    return x

for use_res in [True, False]:
    inp = torch.randn(1, T, D, requires_grad=True)
    out = deep_stack(inp, mask, use_res)
    out.sum().backward()
    print("residuals", str(use_res).ljust(6), " gradient reaching the input :",
          format(inp.grad.abs().mean().item(), '.3e'))

residuals True    gradient reaching the input : 2.419e+00
residuals False   gradient reaching the input : 1.450e-01


with residuals the gradient comes back healthy. without them it is far smaller after only 8 layers, and real models are 12 to 96 layers deep. this is the same vanishing story as the RNN folder, only through depth instead of time.

## parameter count

In [12]:
attn_p = sum(p.numel() for m in [W_q, W_k, W_v, W_o] for p in m.parameters())
ff_p   = sum(p.numel() for p in ff.parameters())
norm_p = sum(p.numel() for m in [norm1, norm2] for p in m.parameters())

print("attention    :", attn_p)
print("feed forward :", ff_p)
print("layer norms  :", norm_p)
print("one block    :", attn_p + ff_p + norm_p)

attention    : 16384
feed forward : 33088
layer norms  : 256
one block    : 49728


### what i learned

- multi head is done with `view` and `transpose`, the heads just become another batch dimension. no loop
- the mask is applied with `masked_fill(-inf)` before the softmax, and it is worth testing rather than assuming. changing the last token and watching the earlier outputs is a clean way to check
- the feed forward part holds twice as many parameters as attention, wich surprised me since attention gets all the attention
- residuals matter a lot even at 8 layers deep
- the block keeps its input shape, thats why they stack